# agenticSeek on Google Colab

This notebook allows you to run the agenticSeek project in a Google Colab environment. This notebook will guide you through the process of setting up and running the project.

In [ ]:
!git clone https://github.com/Fosowl/agenticSeek.git
%cd agenticSeek
!pip install -r requirements.txt -q

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp.git
%cd llama.cpp
!cmake -B build -DCMAKE_BUILD_TYPE=Release
!cmake --build build --config Release -j$(nproc)
%cd ..

In [ ]:
!wget -O llama.cpp/build/bin/model.gguf \
  "https://huggingface.co/lmstudio-community/Llama-3.2-3B-Instruct-GGUF/resolve/main/Llama-3.2-3B-Instruct-Q4_K_M.gguf"

In [ ]:
import subprocess, time, requests
cmd = ["llama.cpp/build/bin/llama-server", "-m", "llama.cpp/build/bin/model.gguf",
       "--host", "0.0.0.0", "--port", "1234", "--jinja", "--threads", "8"]
proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL)
time.sleep(20); assert requests.get("http://localhost:1234/health", timeout=5).ok

In [ ]:
import configparser
cfg = configparser.ConfigParser()
cfg["MAIN"] = {
    "is_local": "True",
    "provider_name": "openai",
    "provider_model": "llama-3.2-3b-instruct-q4_k_m.gguf",
    "provider_server_address": "http://127.0.0.1:1234",
    "work_dir": "/content",
    "headless_browser": "True"
}
with open("config.ini", "w") as f:
    cfg.write(f)

In [ ]:
!python cli.py